# Notebook 08: Comparison to Classical Reconnection Cartoons

This notebook is a **visual bridge**.

Notebook 07 expressed the model in dimensionless, physics-facing language.

Notebook 08 asks:

- what does the 45°-gated current-sheet model look like next to familiar reconnection cartoons?
- can we line up the stages visually without claiming full dynamical equivalence?

This notebook builds side-by-side comparisons between:

- **classical schematic stages**
- **constraint-gated model analogs**

The goal is scientific legibility, not exact solver equivalence.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

THRESHOLD = 1 / np.sqrt(2)

nx, ny = 280, 220
x = np.linspace(-6.0, 6.0, nx)
y = np.linspace(-3.0, 3.0, ny)
X, Y = np.meshgrid(x, y)

dx = x[1] - x[0]
dy = y[1] - y[0]

print("45° threshold =", THRESHOLD)
print("grid shape =", X.shape)

## 1. Shared helpers and lightweight model functions

We use a small subset of the earlier notebook machinery to generate representative model states.

In [ ]:
def normalize_field(Bx, By):
    mag = np.sqrt(Bx**2 + By**2)
    mag = np.where(mag == 0, 1.0, mag)
    return Bx / mag, By / mag

def multimode_sheet_perturbation(X, Y, amps, ks, phases, sigma=0.38):
    out = np.zeros_like(X)
    envelope = np.exp(-(Y**2) / (sigma**2))
    for a, k, p in zip(amps, ks, phases):
        out += a * np.sin(k * X + p)
    return envelope * out

def harris_target(X, Y, delta=0.35):
    return np.tanh(Y / delta)

def laplacian(F, dx, dy):
    return (
        (np.roll(F, 1, axis=0) - 2 * F + np.roll(F, -1, axis=0)) / dy**2 +
        (np.roll(F, 1, axis=1) - 2 * F + np.roll(F, -1, axis=1)) / dx**2
    )

def cross_sheet_cosine(Bx_hat, By_hat, shift=3):
    Bx_up = np.roll(Bx_hat, -shift, axis=0)
    By_up = np.roll(By_hat, -shift, axis=0)

    Bx_dn = np.roll(Bx_hat, shift, axis=0)
    By_dn = np.roll(By_hat, shift, axis=0)

    cos_map = Bx_up * Bx_dn + By_up * By_dn
    cos_map[:shift, :] = np.nan
    cos_map[-shift:, :] = np.nan
    return cos_map

def failed_mask(cos_map, threshold=THRESHOLD):
    return cos_map < threshold

def central_band_mask(mask, Y, half_width=0.8):
    return mask & (np.abs(Y) < half_width)

def make_model_state(delta=0.35, frag_amp=0.10, base_eps=0.06, sigma=0.38, seed=0):
    rng = np.random.default_rng(seed)

    Bx = np.tanh(Y / delta)

    By_background = base_eps * np.sin(1.2 * X) * np.exp(-(Y**2) / (1.0**2))

    ks = [1.0, 2.0, 3.6, 5.0]
    phases = rng.uniform(0, 2 * np.pi, size=len(ks))
    amps = frag_amp * np.array([1.0, 0.75, 0.55, 0.35])
    amps = amps * (1.0 + 0.15 * rng.standard_normal(len(ks)))

    By_fragment = multimode_sheet_perturbation(X, Y, amps, ks, phases, sigma=sigma)
    By = By_background + By_fragment

    Bx_hat, By_hat = normalize_field(Bx, By)
    cos_map = cross_sheet_cosine(Bx_hat, By_hat, shift=3)
    mask = failed_mask(cos_map)
    central = central_band_mask(mask, Y, half_width=0.8)

    return {
        "Bx": Bx,
        "By": By,
        "Bx_hat": Bx_hat,
        "By_hat": By_hat,
        "cos_map": cos_map,
        "mask": mask,
        "central": central,
    }

def evolve_model_state(delta=0.35, frag_amp=0.28, seed=0, n_steps=90, nu=0.0012, alpha=0.25, beta=0.85, gamma=0.18, dt=0.0018):
    rng = np.random.default_rng(seed)

    Bx = np.tanh(Y / delta)
    Bx_target = harris_target(X, Y, delta=delta)

    ks = [1.0, 2.0, 3.6, 5.0]
    phases = rng.uniform(0, 2 * np.pi, size=len(ks))
    amps = frag_amp * np.array([1.0, 0.75, 0.55, 0.35])
    amps = amps * (1.0 + 0.15 * rng.standard_normal(len(ks)))

    By_background = 0.06 * np.sin(1.2 * X) * np.exp(-(Y**2) / (1.0**2))
    seed_pattern = multimode_sheet_perturbation(X, Y, amps, ks, phases, sigma=0.38)
    By = By_background + seed_pattern

    for _ in range(n_steps + 1):
        Bx_hat, By_hat = normalize_field(Bx, By)
        cos_map = cross_sheet_cosine(Bx_hat, By_hat, shift=3)
        act = np.maximum(0.0, THRESHOLD - cos_map)
        act = np.nan_to_num(act, nan=0.0)

        Bx = Bx + dt * (
            nu * laplacian(Bx, dx, dy)
            - alpha * (Bx - Bx_target)
        )

        By = By + dt * (
            nu * laplacian(By, dx, dy)
            + beta * act * seed_pattern
            - gamma * By
        )

        Bx[0, :] = Bx_target[0, :]
        Bx[-1, :] = Bx_target[-1, :]
        By[0, :] = 0.0
        By[-1, :] = 0.0

    Bx_hat, By_hat = normalize_field(Bx, By)
    cos_map = cross_sheet_cosine(Bx_hat, By_hat, shift=3)
    mask = failed_mask(cos_map)
    central = central_band_mask(mask, Y, half_width=0.8)

    return {
        "Bx": Bx,
        "By": By,
        "Bx_hat": Bx_hat,
        "By_hat": By_hat,
        "cos_map": cos_map,
        "mask": mask,
        "central": central,
    }

## 2. Build representative model analogs

We pick four qualitative stages:

1. smooth reversal layer
2. corrugated threshold boundary
3. pocket / plasmoid-like chain
4. regime-map summary

In [ ]:
smooth_state = make_model_state(delta=0.45, frag_amp=0.04, seed=1)
corrugated_state = make_model_state(delta=0.35, frag_amp=0.18, seed=2)
pocket_state = make_model_state(delta=0.28, frag_amp=0.40, seed=3)
dynamic_state = evolve_model_state(delta=0.35, frag_amp=0.28, seed=4, n_steps=110)

print("representative model states ready")

## 3. Classical schematic drawing helpers

These are simple visual cartoons, not external images.

In [ ]:
def setup_cartoon_axis(ax, title):
    ax.set_xlim(-6, 6)
    ax.set_ylim(-3, 3)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(title)

def draw_smooth_sheet(ax):
    setup_cartoon_axis(ax, "Classical schematic: smooth reversal layer")
    for yy in np.linspace(-2.4, -0.45, 5):
        ax.plot(x, 0.15*np.sin(0.35*x) + yy)
    for yy in np.linspace(0.45, 2.4, 5):
        ax.plot(x, -0.15*np.sin(0.35*x) + yy)
    ax.plot(x, 0*x, linestyle="--")
    ax.text(-5.7, 0.2, "sheet")

def draw_corrugated_sheet(ax):
    setup_cartoon_axis(ax, "Classical schematic: corrugated sheet")
    center = 0.28 * np.sin(1.1 * x)
    ax.plot(x, center, linestyle="--")
    for offset in [-1.8, -1.2, -0.6]:
        ax.plot(x, 0.10*np.sin(0.8*x) + center + offset)
    for offset in [0.6, 1.2, 1.8]:
        ax.plot(x, -0.10*np.sin(0.8*x) + center + offset)
    ax.text(-5.7, 0.4, "waviness")

def draw_island_chain(ax):
    setup_cartoon_axis(ax, "Classical schematic: plasmoid-like chain")
    ax.plot(x, 0*x, linestyle="--")
    centers = [-4.0, -1.5, 1.2, 3.8]
    widths = [1.4, 1.1, 1.0, 1.3]
    for c, w in zip(centers, widths):
        e = Ellipse((c, 0.0), width=w, height=0.8, fill=False)
        ax.add_patch(e)
    ax.text(-5.7, 0.6, "islands")

def draw_regime_cartoon(ax):
    setup_cartoon_axis(ax, "Classical schematic: regime organization")
    ax.text(-4.8, 1.8, "smooth")
    ax.text(-0.8, 0.0, "bounded")
    ax.text(2.5, -1.8, "strong")
    ax.arrow(-4.0, 1.5, 2.3, -0.8, head_width=0.15, length_includes_head=True)
    ax.arrow(-0.5, -0.2, 2.5, -1.0, head_width=0.15, length_includes_head=True)
    ax.text(-5.5, -2.5, "more drive")

## 4. Model plotting helpers

In [ ]:
def plot_model_cosine(ax, state, title):
    im = ax.imshow(
        state["cos_map"],
        extent=[x.min(), x.max(), y.min(), y.max()],
        origin="lower",
        aspect="auto"
    )
    ax.contour(
        X, Y, state["cos_map"],
        levels=[THRESHOLD],
        colors="white",
        linewidths=1.5,
        linestyles="--"
    )
    ax.set_title(title)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    return im

def plot_model_central(ax, state, title):
    im = ax.imshow(
        state["central"],
        extent=[x.min(), x.max(), y.min(), y.max()],
        origin="lower",
        aspect="auto"
    )
    ax.set_title(title)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    return im

## 5. Panel A — Smooth sheet comparison

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4.5))

draw_smooth_sheet(axs[0])
im = plot_model_cosine(axs[1], smooth_state, "Constraint-gated model: smooth sheet analog")
fig.colorbar(im, ax=axs[1], label="cross-sheet cosine")

plt.tight_layout()
plt.show()

## 6. Panel B — Corrugated sheet comparison

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4.5))

draw_corrugated_sheet(axs[0])
im = plot_model_cosine(axs[1], corrugated_state, "Constraint-gated model: corrugated threshold boundary")
fig.colorbar(im, ax=axs[1], label="cross-sheet cosine")

plt.tight_layout()
plt.show()

## 7. Panel C — Pocket / plasmoid-like chain comparison

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4.5))

draw_island_chain(axs[0])
im = plot_model_central(axs[1], pocket_state, "Constraint-gated model: pocketed central failed region")
fig.colorbar(im, ax=axs[1], label="central failed mask")

plt.tight_layout()
plt.show()

## 8. Panel D — Dynamic / regime comparison

We use an evolved model state as a dynamic analog.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4.5))

draw_regime_cartoon(axs[0])
im = plot_model_cosine(axs[1], dynamic_state, "Constraint-gated model: evolved fragmented sheet")
fig.colorbar(im, ax=axs[1], label="cross-sheet cosine")

plt.tight_layout()
plt.show()

## 9. Final comparison board

This is the most README / paper-friendly figure in the notebook.

In [ ]:
fig, axs = plt.subplots(4, 2, figsize=(12, 16))

draw_smooth_sheet(axs[0, 0])
plot_model_cosine(axs[0, 1], smooth_state, "Model analog: smooth reversal layer")

draw_corrugated_sheet(axs[1, 0])
plot_model_cosine(axs[1, 1], corrugated_state, "Model analog: corrugated threshold boundary")

draw_island_chain(axs[2, 0])
plot_model_central(axs[2, 1], pocket_state, "Model analog: pocket / plasmoid-like chain")

draw_regime_cartoon(axs[3, 0])
plot_model_cosine(axs[3, 1], dynamic_state, "Model analog: evolved fragmented sheet")

plt.tight_layout()
plt.show()

## 10. Optional vector overlays

These help compare schematic field directions to the model more directly.

In [ ]:
skip = 14

fig, axs = plt.subplots(1, 2, figsize=(12, 4.5))

draw_smooth_sheet(axs[0])

axs[1].quiver(
    X[::skip, ::skip], Y[::skip, ::skip],
    smooth_state["Bx_hat"][::skip, ::skip], smooth_state["By_hat"][::skip, ::skip]
)
axs[1].set_title("Model vector field: smooth analog")
axs[1].set_xlabel("x")
axs[1].set_ylabel("y")

plt.tight_layout()
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4.5))

draw_island_chain(axs[0])

axs[1].quiver(
    X[::skip, ::skip], Y[::skip, ::skip],
    dynamic_state["Bx_hat"][::skip, ::skip], dynamic_state["By_hat"][::skip, ::skip]
)
axs[1].set_title("Model vector field: evolved fragmented analog")
axs[1].set_xlabel("x")
axs[1].set_ylabel("y")

plt.tight_layout()
plt.show()

## 11. Short label summary

These are concise phrases you can reuse in README or paper figure captions.

In [ ]:
labels = [
    "Smooth reversal layer  ↔  failed-band identification",
    "Corrugated sheet       ↔  wavy 45° threshold contour",
    "Island / pocket chain  ↔  disconnected central failed regions",
    "Regime organization    ↔  bounded fragmented current-sheet states",
]

for s in labels:
    print(s)

## 12. Interpretation

Notebook 08 is a communication bridge.

What it supports:

> The 45°-gated current-sheet model reproduces recognizable qualitative stages of reconnection cartoons: smooth sheet, corrugated sheet, pocketed chain structure, and regime organization.

What it does **not** claim:

- exact streamline equivalence
- full dynamical reconnection fidelity
- direct solar-data matching

Best interpretation:

> a visual comparison notebook that maps the geometric current-sheet model onto familiar reconnection-style schematic stages.

This is a good place to stop the first notebook arc and move on to README refinement, paper drafting, or figure export.